# Project Bài 1 · Gợi ý phim cho nhóm người xem để dành

**Dữ liệu.** Bản lưu MovieLens latest-small (`data/raw/movielens.zip`, GroupLens; Harper, Konstan 2015). Người xem được chia theo seed 2027 (`data/project_users_bai01.json`): nhóm bài giảng (300 người, đã dùng trên slide), **nhóm phát triển** (149 người) và **nhóm kiểm tra cuối** (148 người). Hai nhóm sau chưa xuất hiện trên slide.

**Ba mức.** Cơ bản (5 điểm): bước 1–3. Khá (3 điểm): bước 4–5. Nâng cao (2 điểm): chọn một trong A, B, C ở cuối.
**Rubric 10 điểm.** Tái lập 3 · cài đặt và đối chiếu 3 · đánh giá 2 · diễn giải và giới hạn 2. Chấm cách làm và cách đọc kết quả, không chấm con số hit@20 cao hay thấp.

Ô có `TODO` là việc của bạn; ô `TỰ KIỂM` in ✅ khi đúng. Chạy từ trên xuống. **Không** đọc nhóm kiểm tra cuối trực tiếp; chỉ chấm qua `cham_cuoi`, đúng một lần.

In [ ]:
# Thiết lập: chạy ô này trước
from pathlib import Path
import json, time, hashlib, math, zipfile
import numpy as np, pandas as pd, scipy.sparse as sp
from scipy.stats import binomtest
from sklearn.metrics.pairwise import cosine_similarity

ROOT = Path.cwd()
RAW = ROOT / 'data' / 'raw' / 'movielens.zip'
assert RAW.exists(), 'Mở notebook từ thư mục bài (có data/raw/movielens.zip).'
assert hashlib.sha256(RAW.read_bytes()).hexdigest() == '12b11d77fdb2c221daa631e5724104f51ecab1298c0e5f535d56e9ceff02ef8b', 'sai bản lưu dữ liệu'
z = zipfile.ZipFile(RAW)
mv = pd.read_csv(z.open('movies.csv')); rt = pd.read_csv(z.open('ratings.csv')); tags = pd.read_csv(z.open('tags.csv'))
NHOM = json.loads((ROOT / 'data' / 'project_users_bai01.json').read_text())
DEV, _FINAL = NHOM['dev'], NHOM['final']                 # _FINAL chỉ được dùng qua cham_cuoi
idx = {m: i for i, m in enumerate(mv.movieId)}; NM = len(mv); NU = int(rt.userId.max()) + 1
rt['i'] = rt.movieId.map(idx)
POP = np.bincount(rt.i.values, minlength=NM)            # số lượt đánh giá (toàn bộ), dùng phá đồng hạng ở bước 3 như slide

def kiem(ten, dieu_kien, goi_y=''):
    # Ô tự kiểm: in ✅ khi đúng; báo lỗi kèm gợi ý khi sai.
    if not dieu_kien:
        raise AssertionError(f'❌ {ten}. Gợi ý: {goi_y}')
    print(f'✅ {ten}')

_CACHE = {}
def chuan_bi(u):
    # Giao thức của bài (cho sẵn): giấu phim thích (>= 4) được chấm sau cùng theo (thời điểm, movieId);
    # chỉ dùng lượt chấm có thời điểm TRƯỚC thời điểm đó; ứng viên = phim người đó chưa chấm trước thời điểm đó.
    if u in _CACHE:
        return _CACHE[u]
    g = rt[rt.userId == u].sort_values(['timestamp', 'movieId']); l = g[g.rating >= 4]
    T = int(l.timestamp.iloc[-1]); tgt = int(l.i.iloc[-1])
    hist = g[g.timestamp < T]; before = rt[rt.timestamp < T]
    cand = np.ones(NM, bool); cand[hist.i.values] = False
    d = dict(T=T, target=tgt, liked=hist[hist.rating >= 4].i.values, before=before,
             popT=np.bincount(before.i.values, minlength=NM).astype(float), cand=cand)
    assert len(d['liked']) >= 5 and int(before.timestamp.max()) < T
    _CACHE[u] = d
    return d

def xep_20(diem, d):
    # 20 phim ứng viên điểm cao nhất; đồng hạng: phổ biến trước thời điểm giấu giảm dần, rồi movieId tăng dần
    s = np.where(d['cand'], np.round(diem, 12), -np.inf)
    return list(np.lexsort((mv.movieId.values, -d['popT'], -s))[:20])

def goi_y_pho_bien(u):
    d = chuan_bi(u); return xep_20(d['popT'], d)

def hit_at_20(goi_y, nguoi):
    return int(sum(chuan_bi(u)['target'] in goi_y(u) for u in nguoi))

_DAU_VET = ROOT / '.project_bai01_cham_cuoi.json'
def cham_cuoi(cau_hinh, goi_y):
    # Chấm nhóm kiểm tra cuối đúng MỘT lần. goi_y(u) trả 20 chỉ số phim cho người u.
    # Khoá gồm chuỗi cấu hình và "dấu vân tay" của cách gợi ý (mã băm gợi ý cho 20 người đầu nhóm PHÁT TRIỂN, không đụng nhóm cuối).
    # Gọi lại với đúng cấu hình và đúng cách gợi ý thì nhận lại kết quả cũ; khác một trong hai sẽ bị chặn.
    # Đây là cơ chế nhắc kỷ luật, không phải bảo mật: giảng viên đối chiếu tệp dấu vết khi chấm.
    van_tay = hashlib.sha256(np.asarray([goi_y(u) for u in DEV[:20]], np.int64).tobytes()).hexdigest()[:16]
    if _DAU_VET.exists():
        cu = json.loads(_DAU_VET.read_text())
        if cu['cau_hinh'] != cau_hinh or cu['van_tay'] != van_tay:
            raise RuntimeError(f"Nhóm kiểm tra cuối đã được chấm với {cu['cau_hinh']} (dấu vân tay {cu['van_tay']}) lúc {cu['luc']}. Không chấm lại với cấu hình hoặc cách khác.")
        return cu['ket_qua']
    a = np.array([chuan_bi(u)['target'] in goi_y(u) for u in _FINAL]); b = np.array([chuan_bi(u)['target'] in goi_y_pho_bien(u) for u in _FINAL])
    chi_a, chi_b = int((a & ~b).sum()), int((~a & b).sum())
    kq = dict(n=len(_FINAL), hit_chon=int(a.sum()), hit_pho_bien=int(b.sum()), chi_chon_trung=chi_a, chi_pho_bien_trung=chi_b,
              p_dau=float(binomtest(chi_a, chi_a + chi_b, 0.5).pvalue) if chi_a + chi_b else 1.0)
    _DAU_VET.write_text(json.dumps(dict(cau_hinh=cau_hinh, van_tay=van_tay, ket_qua=kq, luc=time.strftime('%Y-%m-%d %H:%M'))))
    return kq
print(len(mv), 'phim ·', len(rt), 'lượt ·', 'nhóm phát triển', len(DEV), '· nhóm kiểm tra cuối', len(_FINAL))

## Bước 1 (cơ bản) · Hàm cosine an toàn

In [ ]:
def cosine_an_toan(u, v):
    # TODO: trả cosine của hai vector; None nếu có vector 0; ValueError nếu khác số chiều, rỗng, hoặc có NaN/vô cực
    raise NotImplementedError

In [ ]:
# TỰ KIỂM · bước 1
kiem('cos((1;2),(2;1)) = 0,8', abs(cosine_an_toan([1, 2], [2, 1]) - 0.8) < 1e-12)
kiem('không đổi khi nhân số dương, đổi dấu khi nhân −1', abs(cosine_an_toan([3, 6], [2, 1]) - 0.8) < 1e-12 and abs(cosine_an_toan([-1, -2], [2, 1]) + 0.8) < 1e-12)
kiem('vector 0 cho None (không xác định)', cosine_an_toan([1, 0, 1], [0, 0, 0]) is None, 'kiểm độ dài trước khi chia')
for bad in ([1, 2], [np.nan, 0, 1], [np.inf, 0, 1], []):
    try:
        cosine_an_toan([1, 0, 1], bad); chan = False
    except ValueError:
        chan = True
    kiem(f'chặn đầu vào hỏng {bad}', chan, 'raise ValueError')

## Bước 2 (cơ bản) · Ma trận phim × 19 nhãn

In [ ]:
# TODO: NHAN = danh sách tập nhãn của từng phim (bỏ '(no genres listed)'); AX = 19 nhãn xếp theo bảng chữ cái;
#       X = ma trận 0/1, hàng = phim (đúng thứ tự mv), cột = nhãn theo AX
raise NotImplementedError

In [ ]:
# TỰ KIỂM · bước 2
kiem('ma trận 9742 × 19, đúng thứ tự trục', X.shape == (9742, 19) and AX[0] == 'Action' and AX[-1] == 'Western')
kiem('34 phim không có nhãn (vector 0)', int((X.sum(1) == 0).sum()) == 34)
kiem('The Matrix có 3 nhãn: Action, Sci-Fi, Thriller', [AX[j] for j in np.flatnonzero(X[idx[2571]])] == ['Action', 'Sci-Fi', 'Thriller'])
kiem('cột Drama có 4361 phim', int(X[:, AX.index('Drama')].sum()) == 4361)

## Bước 3 (cơ bản) · Top 10 phim giống, có quy tắc đồng hạng; đối chiếu scikit-learn

In [ ]:
def top_k(movie_id, k=10):
    # TODO: trả (danh sách k movieId, danh sách k cosine, số phim đồng điểm với phim thứ k); trả None nếu phim truy vấn là vector 0
    #       chuẩn hoá L2 từng hàng của X; bỏ chính phim truy vấn và vector 0; làm tròn điểm 12 chữ số;
    #       đồng hạng: POP giảm dần, rồi movieId tăng dần
    raise NotImplementedError

In [ ]:
for m in [1, 296, 79132]:
    ids, cs, ndd = top_k(m)
    print(mv.title[idx[m]], '→', [mv.title[idx[i]] for i in ids[:5]], '· số phim đồng điểm với phim thứ 10:', ndd)

In [ ]:
# TỰ KIỂM · bước 3
ids, cs, ndd = top_k(2571)
kiem('The Matrix: top 10 khớp slide, 42 phim đồng điểm 1', ids == [1240, 541, 3527, 172, 45499, 5903, 1882, 34319, 68319, 8371] and ndd == 42)
for m in [1, 296, 79132]:
    ids, cs, ndd = top_k(m)
    sk = cosine_similarity(X[[idx[i] for i in ids]], X[[idx[m]]]).ravel()
    kiem(f'phim {m}: 10 phim, không chứa chính nó, điểm giảm dần, khớp scikit-learn', len(ids) == 10 and m not in ids and all(a >= b for a, b in zip(cs, cs[1:])) and np.allclose(sk, cs))

**Câu hỏi.** Với phim nào trong ba phim, quy tắc đồng hạng quyết định phần lớn top 10? Vì sao Inception không có phim nào cosine 1 mà vẫn có nhiều phim đồng điểm ở vị trí thứ 10?

## Bước 4 (khá) · Gợi ý cho người xem: nhãn, hành vi, mốc phổ biến

Dùng `chuan_bi(u)` (cho sẵn, đúng giao thức của bài): chỉ lượt chấm trước thời điểm giấu. `goi_y_pho_bien` và `xep_20` cho sẵn.

In [ ]:
def goi_y_nhan(u):
    # TODO: hồ sơ = tổng vector đơn vị (19 nhãn) của các phim đã thích trong chuan_bi(u)['liked'];
    #       điểm của mỗi phim = cosine(hồ sơ, phim); trả xep_20(điểm, chuan_bi(u))
    raise NotImplementedError

def goi_y_hanh_vi(u):
    # TODO: chỉ dùng chuan_bi(u)['before'] (lượt chấm trước thời điểm giấu): phim = vector 0/1 "ai đã chấm >= 4";
    #       điểm của phim m = tổng cosine giữa m và từng phim đã thích; trả xep_20(điểm, chuan_bi(u))
    raise NotImplementedError

In [ ]:
t0 = time.time()
hit_dev = {ten: hit_at_20(f, DEV) for ten, f in (('nhan', goi_y_nhan), ('pho_bien', goi_y_pho_bien), ('hanh_vi', goi_y_hanh_vi))}
def theo_cap(f, g, nguoi):
    a = np.array([chuan_bi(u)['target'] in f(u) for u in nguoi]); b = np.array([chuan_bi(u)['target'] in g(u) for u in nguoi])
    x, y = int((a & ~b).sum()), int((~a & b).sum())
    return x, y, (float(binomtest(x, x + y, 0.5).pvalue) if x + y else 1.0)
cap_dev = theo_cap(goi_y_hanh_vi, goi_y_pho_bien, DEV)
print('hit@20 nhóm phát triển', hit_dev, f'/ {len(DEV)} · hành vi so với phổ biến theo cặp:', cap_dev, f'· {time.time() - t0:.0f} giây')

In [ ]:
# TỰ KIỂM · bước 4
u0 = DEV[0]; d0 = chuan_bi(u0)
for ten, f in (('nhãn', goi_y_nhan), ('hành vi', goi_y_hanh_vi)):
    g = f(u0)
    kiem(f'{ten}: 20 phim, đều là phim chưa chấm trước thời điểm giấu', len(g) == 20 and all(d0['cand'][i] for i in g))
kiem('mốc phổ biến trên nhóm phát triển: 8 người (mã cho sẵn)', hit_dev['pho_bien'] == 8)

## Bước 5 (khá) · Ghi quy tắc chọn, rồi chấm nhóm kiểm tra cuối một lần

Viết quy tắc chọn của bạn trong ô dưới **trước** khi chạy ô chấm cuối (lời giải mẫu: chọn cách có hit@20 cao nhất trên nhóm phát triển, nếu hơn mốc phổ biến ít nhất 3 người; ngược lại chọn mốc phổ biến). `cham_cuoi` ghi lần chấm (chuỗi cấu hình và dấu vân tay của cách gợi ý) vào `.project_bai01_cham_cuoi.json`; chạy lại notebook với cùng cấu hình và cùng cách thì nhận lại kết quả cũ, đổi một trong hai sẽ bị chặn. Đây là cơ chế nhắc kỷ luật, không phải bảo mật. Ô tự kiểm không cho biết đáp số của nhóm này.

In [ ]:
# TODO: viết quy tắc chọn của BẠN trước khi chạy ô chấm cuối (dựa trên hit_dev và cap_dev ở bước 4).
#       Có thể thêm cách nâng cao của bạn vào CACH trước khi chọn. Sau khi chấm cuối, không sửa ô này.
CACH = {'nhan': goi_y_nhan, 'pho_bien': goi_y_pho_bien, 'hanh_vi': goi_y_hanh_vi}
QUY_TAC = '...'
chon = ...
raise NotImplementedError

In [ ]:
cau_hinh = f'chon={chon}'
kq = cham_cuoi(cau_hinh, CACH[chon])
print(f"nhóm kiểm tra cuối ({kq['n']} người): cách đã chọn {kq['hit_chon']} · mốc phổ biến {kq['hit_pho_bien']}")
print(f"theo cặp: chỉ cách đã chọn trúng {kq['chi_chon_trung']} · chỉ phổ biến trúng {kq['chi_pho_bien_trung']} · kiểm định dấu p = {kq['p_dau']:.3g}")

In [ ]:
# TỰ KIỂM · bước 5
kiem('đã ghi quy tắc chọn và chọn một cách có trong CACH', isinstance(QUY_TAC, str) and len(QUY_TAC) > 20 and chon in CACH)
kiem('kết quả là số người hợp lệ', 0 <= kq['hit_chon'] <= kq['n'] and 0 <= kq['hit_pho_bien'] <= kq['n'] and kq['n'] == 148)
kiem('số người bất đồng khớp hiệu số trúng', kq['chi_chon_trung'] - kq['chi_pho_bien_trung'] == kq['hit_chon'] - kq['hit_pho_bien'])
try:
    cham_cuoi('chon=khac', CACH[chon]); chan = False
except RuntimeError:
    chan = True
kiem('hàm chặn chấm lần hai với cấu hình khác', chan)
try:
    cham_cuoi(cau_hinh, lambda u: list(range(20))); chan2 = False
except RuntimeError:
    chan2 = True
kiem('hàm chặn cả khi giữ chuỗi cấu hình nhưng đổi cách gợi ý', chan2)

## Báo cáo một trang (nộp cùng notebook)

1. **Dữ liệu và giao thức.** Ba nhóm người, cách chia; vì sao chỉ dùng lượt chấm trước thời điểm giấu; thời điểm trong MovieLens là lúc *nhập* điểm.
2. **Cài đặt và đối chiếu.** Cosine an toàn (vector 0), khớp scikit-learn; top 10 của ba phim và số phim đồng điểm.
3. **Đánh giá.** hit@20 ba cách trên nhóm phát triển; so theo cặp với mốc phổ biến; quy tắc chọn đã ghi; kết quả nhóm cuối (một lần).
4. **Diễn giải và giới hạn.** Không đọc cosine như xác suất; hit@20 là tỷ lệ người; ít nhất ba giới hạn (cỡ nhóm, một phim giấu mỗi người, ngưỡng thích, nhãn thô, phiên nhập điểm).

## Nâng cao (chọn một; không có ô tự kiểm, chấm theo rubric)

- **A · Trọng số IDF cho nhãn** (slide mở rộng): so hit@20 trên nhóm phát triển có và không có trọng số.
- **B · Vector từ `tags.csv`:** phim = vector 0/1 các thẻ người dùng gắn; tìm phim giống The Matrix; nêu giới hạn (nhiều phim không có thẻ).
- **C · Trộn nhãn và hành vi** với α chọn trên nhóm phát triển; báo cả đường hit@20 theo α, không chỉ giá trị tốt nhất.

In [ ]:
# Nâng cao A (tuỳ chọn): trọng số IDF cho nhãn, so trên nhóm phát triển

In [ ]:
# Nâng cao B (tuỳ chọn): vector thẻ từ tags.csv, phim giống The Matrix, giới hạn

In [ ]:
# Nâng cao C (tuỳ chọn): trộn nhãn và hành vi, chọn α trên nhóm phát triển